# Capstone — Govern a Federated Retail Data Platform

**Cardinal Outfitters** is a mid-sized outdoor & athletic gear retailer that grew by
acquisition and now runs as a federation of four loosely-coupled domain teams on a shared
data platform:

- **E-commerce Orders** — purchase events, returns
- **Customer Identity & Loyalty** — profiles, consent, loyalty (the customer system of record)
- **Inventory & Fulfillment** — products, suppliers, shipments, per-shipment carbon
- **Marketing & Campaigns** — campaign membership, segments, marketing consent

No enforced contracts, no shared metadata standards, partial quality coverage, no lineage.
Two pressures land in the same month: a **GDPR audit notice** and a new **CSRD/ESG
reporting mandate** requiring quarterly sustainability disclosures with per-order carbon
traceable to source.

**You are the data architect** bringing this platform under federated governance. This is
a judgment project: you run provided tooling, read its output, make governance decisions,
and write decision-oriented artifacts. The grade is on the decisions, not on engineering
throughput.

## How this notebook works
- **Provided cells** (setup, audits, pipelines) are ready to run — just execute them.
- **`TODO(you)` code cells** and **deliverable markdown cells** are yours to complete.
- Run top-to-bottom. Re-running rebuilds a clean platform from scratch.

## What you submit
This completed notebook, with every `TODO` filled and every deliverable written, ending
with your 1–2 page **Architect's Brief** (final section).


### A note on the backend (read me)

This project is meant to be run against **AWS in your Vocareum lab**. Three steps perform
real cloud operations:

- **Step 2 (MP2)** reads and updates the **AWS Glue Data Catalog**.
- **Step 7 (MP7)** configures **AWS Lake Formation** (column / row / tag grants) and
  verifies enforcement with **Amazon Athena**, assuming each persona role.
- **Step 9 (MP9)** audits the live Glue + Lake Formation state and you remediate in AWS.

`make_catalog()` returns the **live AWS engine** when the lab is provisioned
(`lf_backend()` reports `live`). If AWS is not available it transparently falls back to an
equivalent **local engine** so the notebook still runs end-to-end while you author — but
the **graded path is AWS**. The remaining steps (MP4 quality, MP5 master data, MP6 lineage,
MP8 reporting) operate on the same dataset either way. `lf_backend()` tells you which path
is active; aim to complete the project with it reporting `live`.


## Step 0 — Set up the platform
Run this cell. It connects to the platform (live AWS Glue + Lake Formation when your Vocareum lab is provisioned, else the local engine), loads the catalog, and reports which backend is active.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

from governance_toolkit import (
    make_catalog, lf_backend, verify_step7,
    validate_contract, base_contract, apply_adjudications, PERSONAS,
)

gc = make_catalog("cardinal.db")   # live AWS engine if provisioned, else local fallback
print("Backend:", lf_backend(), "->", type(gc).__name__)
print("Personas:", list(PERSONAS))

overview = pd.DataFrame(
    [{"table": t, "domain": gc.catalog.table(t).domain,
      "rows": gc.query(f"SELECT COUNT(*) c FROM {t}").c[0]} for t in gc.tables()]
)
overview


## Step 1 — Choose a governance model (MP1)

Cardinal's four domains own their pipelines and definitions and won't accept a central
team rewriting them — but GDPR and CSRD create obligations that cross every domain. Decide
how to govern: **centralized**, **decentralized**, or **federated**, and which decisions
stay central even within your choice.

**Deliverables:** (1) a governance-model decision, (2) a RACI across the four domains, and
(3) a Mermaid topology of the federated model.


In [ ]:
# Provided: the federation at a glance — each domain and the data products it owns.
pd.DataFrame([{"domain": gc.catalog.table(t).domain, "table": t,
               "owner": gc.catalog.table(t).owner or "(no owner set)"}
              for t in gc.tables()]).sort_values("domain").reset_index(drop=True)


### 1a. Governance model — decision

**Choice: federated governance with a thin central policy core.**

Cardinal grew by acquisition; each domain owns its pipelines and its definitions, and
will resist a centralized team that rewrites them — centralized governance would stall on
political friction and become a bottleneck for four independently-shipping teams.
Fully decentralized governance is what they have today, and it produced the failure mode
under audit: no shared consent system of record, no common metadata standard, no lineage.

Federated governance keeps **domain ownership of data products** while making a small set
of decisions **non-negotiable and central**:

- **Classification taxonomy & PII tagging standard** — central, because GDPR exposure is
  platform-wide and inconsistent tagging is what breaks subject-access response.
- **Consent system of record** — central; today loyalty and marketing disagree (~20% of
  records), which is a direct GDPR risk.
- **Contract schema & required governance fields** — central schema, domain-authored content.
- **Shared metric definitions** (e.g., per-order carbon) — central, because CSRD requires
  one auditable number across domains.

Everything else — schema evolution within a product, pipeline implementation, quality
rule authoring — stays with the domains. This is the minimum centralization that satisfies
the two regulators without seizing day-to-day control from the teams.


### 1b. RACI (governance functions × domains)

| Governance function | E-comm Orders | Customer Identity | Inventory/Fulfillment | Marketing | Accountable (single) |
|---|---|---|---|---|---|
| PII classification & tagging | R | R | R | R | **Central Data Governance Lead (A)** |
| Consent system of record | C | R | I | C | **Customer Identity Lead (A)** |
| Data contracts (author) | R | R | R | R | **Owning domain lead (A, per product)** |
| Data quality SLOs | R | R | R | R | **Owning domain lead (A)** |
| Lineage instrumentation | C | C | R | C | **Platform Engineering (A)** |
| Carbon metric definition | I | I | R | C | **CSRD Reporting Owner (A)** |
| Access policy (Lake Formation) | C | R | C | C | **Central Data Governance Lead (A)** |

**Enforcement authority:** the Central Data Governance Lead can block a data product from
publishing if it fails the classification or contract gate. **Escalation:** unresolved
cross-domain conflicts (e.g., consent) escalate to a Data Governance Council chaired by the
Lead, with the two domain leads as voting members.


### 1c. Federated topology

```mermaid
graph TD
    GC[Central Governance Core<br/>classification • consent SoR • contract schema • metrics]
    subgraph Domains
        D1[E-commerce Orders]
        D2[Customer Identity & Loyalty]
        D3[Inventory & Fulfillment]
        D4[Marketing & Campaigns]
    end
    GC -- standards & gates --> D1 & D2 & D3 & D4
    D2 -- consent SoR --> D4
    D1 -- order events --> D3
    D3 -- carbon source --> R[CSRD Quarterly Report]
    D1 -- order carbon --> R
```


## Step 2 — Reconcile the catalog (MP2)

Metadata is your first control surface. Connect to the **AWS Glue Data Catalog**, audit it
for governance gaps (missing owner / classification / retention, and untagged PII columns),
**fix the gaps in the catalog**, then **re-run the audit to verify** your fixes landed.
Finally, write a reconciliation memo.


In [ ]:
# Reads the live AWS Glue Data Catalog (or the local catalog when running offline).
audit = gc.catalog_audit()
print("Governance gaps found:", len(audit))
audit


In [ ]:
gc.semantic_conflicts()

In [ ]:
# Summarize the gaps by domain and gap type so the memo can prioritize.
summary = (audit.groupby(["domain", "gap_type"]).size()
           .reset_index(name="count").sort_values("count", ascending=False))
summary


**Now fix the gaps in the catalog.** Use the provided helpers to write governance metadata
back to AWS Glue, then re-audit to confirm. `gc.set_table_metadata(table, owner=...,
classification=..., retention=...)` updates table-level Glue parameters;
`gc.tag_column(table, column, classification)` tags a column (e.g., the untagged PII column).


In [ ]:
# Remediate every gap the audit found, writing the metadata back to the Glue catalog.
gc.set_table_metadata("orders", owner="orders-team@cardinal.example", retention="P7Y")
gc.set_table_metadata("order_returns", owner="orders-team@cardinal.example",
                      classification="SENSITIVE", retention="P7Y")
gc.set_table_metadata("product", classification="SENSITIVE", retention="P5Y")
gc.set_table_metadata("shipment", owner="inventory-fulfillment@cardinal.example")
gc.set_table_metadata("campaign_membership", owner="marketing@cardinal.example",
                      classification="PII", retention="P3Y")   # ungoverned PII table
gc.tag_column("customer", "phone", "PII")                      # untagged PII column
gc.tag_column("order_returns", "reason", "PUBLIC")
gc.tag_column("product", "sustainability_class", "PUBLIC")
print("Remediation written to the catalog.")


In [ ]:
# Provided: re-audit to verify your fixes. This should drop to 0 gaps when complete.
re_audit = gc.catalog_audit()
print("Gaps before:", len(audit), "-> after remediation:", len(re_audit))
assert len(re_audit) == 0, "still gaps remaining — fix the rows below before continuing"
re_audit


### 2. Reconciliation memo

**Gaps (must-fix for the GDPR audit):**
- **Untagged PII column `customer.phone`** — a personal-data column with no classification.
  This is the highest-severity gap: it is exactly what a subject-access request would miss.
- **`campaign_membership`** carries `cust_ref` (customer email = PII) and its own consent
  flag but has **no owner, no classification, no retention** — an ungoverned PII table.
- Missing **owner** on `orders`, `order_returns`, `shipment`, `campaign_membership`;
  missing **retention** on `orders`, `product`, `order_returns`, `campaign_membership`;
  missing **classification** on `product`, `order_returns`.

**Cross-domain semantic conflicts:**
1. **Customer identity key** — loyalty keys on `customer_id` (integer), marketing on
   `cust_ref` (email). No shared join key; cross-domain customer resolution must match on
   email (Step 5).
2. **Consent** — `customer.marketing_consent` vs `campaign_membership.mkt_consent` disagree
   ~20% of the time. Two systems of record for consent is a direct GDPR violation risk.
3. **Carbon unit** — `shipment.carbon_kg` mixes kg and a grams slice; breaks CSRD
   traceability (Step 8).

**Recommended standards (tie to MP9 governance view):** mandatory `owner`,
`classification`, `retention` on every table before publish; a platform PII tag standard
applied to `customer.phone`, `customer.email`, `campaign_membership.cust_ref`; and a single
designated consent system of record (loyalty) that marketing must read, not duplicate.


## Step 3 — Encode obligations in a data contract (MP3)

A data contract turns governance intent into something a pipeline can validate. Start from
the provided ungoverned `orders` contract, extend it with a **governance block**
(PII tags, retention, quality SLOs), validate it with the provided validator, then explain
how a policy change propagates.


In [ ]:
base = base_contract()
ok, errs = validate_contract(base)
print("Base contract valid?", ok)
print("Validator errors:", errs)   # it fails: no governance block yet
base


In [ ]:
# Extend the contract with the required governance block.
contract = base_contract()
for col in contract["columns"]:
    col["pii"] = col["name"] in ("customer_id",)   # FK to a PII entity
contract["governance"] = {
    "pii_tags": ["customer_id"],
    "retention": "P7Y",
    "quality_slos": [
        {"dimension": "completeness", "threshold": 0.99},
        {"dimension": "validity", "threshold": 0.98},
        {"dimension": "accuracy", "threshold": 0.995},
    ],
}
ok, errs = validate_contract(contract)
print("Extended contract valid?", ok, "| errors:", errs)
assert ok, "fix the governance block until the validator passes"
contract["governance"]


### 3. Policy-propagation write-up

Adding `retention: P7Y` and PII tags to the `orders` contract is not a local change.

- **Producers** (the Orders domain pipeline) must now emit records that satisfy the quality
  SLOs and must enforce 7-year retention/deletion — a code and storage-lifecycle change.
- **Consumers** that join `orders.customer_id` to customer data inherit the PII obligation:
  the join result is PII even though `orders` alone looks benign, so downstream products
  (e.g., `customer_360`, `marketing_audience`) must re-classify and re-tag.
- **Propagation mechanism:** a contract change **version-bumps** the contract (1.0.0 →
  1.1.0). Every downstream contract that depends on `orders` is flagged for review; the
  central schema gate refuses to publish a downstream product whose contract references an
  outdated upstream version. The remediation sequence is upstream-first: fix `orders`,
  re-validate, then walk the lineage (Step 6) downstream, re-validating each dependent
  product in topological order.


## Step 4 — Interpret quality & plan remediation (MP4)

Run the provided quality suite (Great-Expectations-shaped results across six dimensions).
For each failure, decide a remediation strategy: **source** (fix at the producer),
**downstream** (fix in transform), or **quarantine** (isolate until fixed).


In [ ]:
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q


In [ ]:
# Provided: a dimension-level rollup.
q.groupby("dimension").agg(checks=("success", "size"),
                           failing=("success", lambda s: int((~s).sum())),
                           worst_pct=("unexpected_percent", "max")).reset_index()


In [ ]:
# Classify each FAILING expectation as source / downstream / quarantine.
remediation = {
    "product.category":             "source",       # producer must populate mandatory attr
    "product.sustainability_class": "source",       # CSRD attr; fix at inventory domain
    "customer.email":               "quarantine",   # invalid PII contact; isolate from sends
    "customer.phone":               "quarantine",   # invalid contact; isolate
    "orders.amount":                "source",        # negative amounts = upstream data error
    "shipment.carbon_kg":           "downstream",    # unit normalization in the transform
    "customer.created_ts":          "downstream",    # backfill/flag stale at load
    "campaign_membership.mkt_consent": "source",     # consent must come from the SoR, not be duplicated
}
fails = q[~q["success"]].assign(
    key=lambda d: d["table"] + "." + d["column"],
)
fails["strategy"] = fails["key"].map(remediation)
fails[["key", "dimension", "unexpected_percent", "strategy"]]


### 4. Prioritized remediation plan

**Priority 1 — quarantine now (GDPR + active harm):** `customer.email` and
`customer.phone` validity failures. Invalid contact data on PII columns means sends fail
*and* a subject-access request returns garbage. Quarantine these records out of marketing
audiences until the contact data is corrected, and raise a contract validity SLO (0.98) so
the producer is accountable.

**Priority 2 — fix at source (mandatory-attribute & integrity gaps):**
`product.category`, `product.sustainability_class`, `orders.amount` (negatives are an
upstream integrity error), and `campaign_membership.mkt_consent` (must be read from the
loyalty consent SoR, not independently captured). These tie to the MP3 contract SLOs:
producers must meet completeness ≥ 0.99 before publish.

**Priority 3 — downstream fixes:** `shipment.carbon_kg` unit normalization belongs in the
reporting transform (Step 8), and `customer.created_ts` staleness can be flagged at load.

**Drift watch:** the consent mismatch (consistency) and carbon unit (accuracy) are not
one-time fixes — they recur every load. Recommend contract revisions that assert these as
ongoing SLOs and feed them to the Step 9 dashboard.


## Step 5 — Master-data stewardship (MP5)

There is no shared customer key across domains, so duplicates exist. Run the provided
resolution workflow. High-confidence pairs auto-merge; the **ambiguous** band needs your
stewardship judgment. Adjudicate each, then document survivorship.


In [ ]:
m = gc.resolve_customers()
print("auto-merge:", (m.pairs.band == "auto-merge").sum(),
      "| ambiguous (your queue):", len(m.ambiguous),
      "| rejected:", (m.pairs.band == "reject").sum())
m.ambiguous[["id_a", "name_a", "email_a", "id_b", "name_b", "email_b", "score"]]


In [ ]:
# Adjudicate each ambiguous pair by its row index: "accept" (same person) or "reject".
# These six are all same-name + email-variant (domain swap or inserted digit) of one
# person — accept all six as merges.
decisions = {0: "accept", 1: "accept", 2: "accept",
             3: "accept", 4: "accept", 5: "accept"}
outcome = apply_adjudications(m, decisions)
print({k: v for k, v in outcome.items() if k != "merges"})
outcome["merges"]


### 5. Stewardship decision log

| Pair | Signal | Decision | Rationale |
|---|---|---|---|
| same name, domain-swapped email | name match + email-local match | **accept** | Same person re-registered with a different mail provider; loyalty + the new record are one identity. |
| same name, inserted-digit email | name match + email-local match | **accept** | Typo/auto-suffix on re-entry; clearly the same person. |
| whitespace name variant | normalized-name match | **accept** | Cosmetic name difference only. |

**Survivorship rule:** the **lowest `customer_id` wins** as the golden record, because the
loyalty domain is the designated customer system of record and lower ids are the original
registrations. Non-key attributes take the most-recent non-null value.

**False-positive / false-negative handling:** I accepted only pairs with *both* a name
signal and an email-local signal; pairs sharing only a last name (the rejected band) are
left unmerged to avoid false merges (merging two real people is worse than missing a
duplicate, under GDPR — you would expose one person's data to another). The residual risk
is false negatives where a person changed both name and email; those need a deterministic
key (loyalty id) propagated to marketing, which is the real structural fix.

**Cross-domain stewardship:** the root cause is marketing keying on email instead of the
loyalty `customer_id`. The durable remediation is to push the loyalty id into the marketing
domain as the shared key, eliminating email-based resolution entirely.


## Step 6 — Read lineage & specify monitoring (MP6)

The platform is pre-instrumented (OpenLineage-shaped events). Read the lineage, trace a
simulated breach to its downstream impact, then specify monitoring.


In [ ]:
print("Pipeline jobs:")
display(gc.lineage.events())
print("\nDatasets:", sorted(gc.lineage.datasets()))
print("\nMermaid (paste into any Mermaid renderer):\n")
print(gc.lineage.to_mermaid())


In [ ]:
# Provided: a simulated breach at the shipment source (a carbon-unit error was introduced).
breach = gc.lineage.simulate_breach("shipment")
breach


In [ ]:
# Trace the customer-PII blast radius: if `customer` is breached, what is exposed downstream?
pii_impact = gc.lineage.trace_downstream("customer")
print("Downstream of a customer breach:", pii_impact)
# And confirm the upstream sources of the regulated report:
report_sources = gc.lineage.trace_upstream("quarterly_sustainability")
print("Upstream sources of the CSRD report:", report_sources)


### 6. Monitoring specification

| Signal | Source dataset/job | Threshold | Owner | Ties to |
|---|---|---|---|---|
| Carbon unit out of range | `build_order_carbon` | any `carbon_kg` > 100 | Inventory/Fulfillment | accuracy SLO; CSRD report |
| Consent mismatch rate | `build_customer_360` | > 1% loyalty≠marketing | Customer Identity | consistency SLO; GDPR |
| Freshness of `quarterly_sustainability` | report job | > 24h since last run | CSRD Reporting Owner | report SLA |
| Null mandatory attrs (`product.category`) | product producer | > 0% | Inventory/Fulfillment | completeness SLO |
| Lineage event missing | any job | run with no OpenLineage event | Platform Engineering | governance audit (MP9) |

**Breach impact reading:** a change at `shipment` propagates to `order_carbon` and then
`quarterly_sustainability` — i.e., a source carbon error silently corrupts the regulated
disclosure with no intervening gate. A `customer` breach exposes `customer_360` and
`marketing_audience` downstream. **Gap the current signals miss:** there is no monitor on
the `customer_id` ↔ `cust_ref` join quality, so a resolution regression would not alarm —
recommend adding match-rate monitoring on `build_customer_360`.


## Step 7 — Configure governed access (MP7) — the hands-on step

This is the primary hands-on AWS step. Four personas must end up with exactly the access
below. You apply the grants via the provided helpers — under the live backend these make
**real AWS Lake Formation** grants (column, row, tag, and metadata-only) — then run the
verification cell, which **assumes each persona role and queries the governed table in
Amazon Athena** to prove enforcement.

| Persona | Intended access | LF mode |
|---|---|---|
| `co_marketing_analyst` | Customer **minus** `email`, `phone` | column-level |
| `co_cs_rep` | All columns, **rows where `region='EU'`** only | row-level |
| `co_data_steward` | Everything tagged `governed=true` | tag-based (LF-TBAC) |
| `co_auditor` | Schema + tags only; **no row data** | metadata-only |


### 7a. Policy spec (author before you configure)

- **`co_marketing_analyst`** targets campaigns and must never see raw contact PII. Grant
  SELECT on `customer` **excluding `email`, `phone`**. Order/segment aggregates allowed;
  contact columns denied at the column level.
- **`co_cs_rep`** handles EU tickets only: full customer detail but **only `region='EU'`
  rows**. Implement as a data-cell (row) filter; no access to other regions' customers.
- **`co_data_steward`** owns quality across the customer domain: full access to everything
  tagged `governed=true`, granted by **LF-Tag**, not per-table, so new governed tables are
  covered automatically.
- **`co_auditor`** verifies posture and must **not** read customer data: **DESCRIBE /
  schema + classification only**, `SELECT` denied. Least privilege — the auditor proves
  governance exists without ever touching a personal record.


In [ ]:
# Provided: confirm the personas and current (empty) grant state.
print("Backend:", lf_backend())
gc.grants_summary()


In [ ]:
# Apply the four grants.
# 1) column-level: analyst sees customer minus PII
gc.grant_columns("co_marketing_analyst", "customer", exclude=["email", "phone"])

# 2) row-level: cs_rep sees only EU rows
gc.create_row_filter("eu_only", "customer", "region = 'EU'")
gc.grant_rows("co_cs_rep", "customer", "eu_only")

# 3) tag-based: tag the customer table governed=true, grant steward by tag
gc.assign_tag("customer", "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")

# 4) metadata-only: auditor gets DESCRIBE, no SELECT
gc.grant_describe_only("co_auditor")

gc.grants_summary()


In [ ]:
# Provided verification: query the governed table as each persona and assert enforcement.
results = verify_step7(gc)
print("All checks pass?", (results["result"] == "PASS").all())
results


**Now read the provided Terraform.** Open `terraform/governed-data-product/main.tf`. It
packages the same chain you just configured by hand — S3 location → Glue table → Lake
Formation grant → IAM consumer role — as a reusable module. Write the read-and-explain note
below.


### 7b. Terraform read-and-explain + sample IAM

The `governed-data-product` module provisions one governed data product as a four-link
chain:

1. **`aws_s3_bucket.data`** — the physical storage for the product, tagged with domain and
   `GovernedBy = lake-formation`.
2. **`aws_lakeformation_resource.data`** — registers that S3 location with Lake Formation so
   **LF, not raw IAM/S3, mediates reads**. This is the linchpin: without it, IAM S3
   permissions would bypass fine-grained control.
3. **`aws_glue_catalog_table.table`** — the catalog entry, carrying `owner`,
   `classification`, `retention` parameters (the same governance metadata Step 2 audits) and
   an LF-Tag `governed=true` for tag-based grants.
4. **`aws_iam_role.consumer` + policy** — a consumer identity that can run Athena and read
   the **results bucket only**; it has **no `s3:GetObject` on the data bucket**, so its only
   path to data is LF-mediated. `aws_lakeformation_permissions.consumer_columns` then grants
   SELECT while **excluding the PII columns** — column-level deny, declaratively.

The module is the repeatable, audited version of my manual Step 7 grants: same enforcement,
but version-controlled and applied identically to every new data product.

**Sample IAM policy for a persona role (no direct data-bucket access):**
```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Effect": "Allow",
     "Action": ["athena:StartQueryExecution","athena:GetQueryResults",
                "glue:Get*","lakeformation:GetDataAccess"],
     "Resource": "*"},
    {"Effect": "Allow",
     "Action": ["s3:GetObject","s3:PutObject"],
     "Resource": "arn:aws:s3:::cardinal-athena-results/*"}
  ]
}
```
The deliberate omission of `s3:GetObject` on `cardinal-*-data` is what forces every read
through Lake Formation.


## Step 8 — Verify cross-domain reporting auditability (MP8)

The CSRD quarterly carbon disclosure spans Orders + Inventory. Run the provided pipeline,
then verify you can trace a reported figure back to source — and find where traceability
breaks.


In [ ]:
rep = gc.run_sustainability_report()
print("Quarterly disclosure:")
display(rep.quarterly)
print("\nReconciliation (disclosed vs recomputed-from-source):")
rep.reconciliation


In [ ]:
# Trace one reported quarter back to its source shipments, and quantify the unit defect.
q0 = rep.quarterly.iloc[0]["quarter"]
detail = rep.per_order[rep.per_order["quarter"] == q0]
suspect = detail[detail["carbon_kg"] > 100]    # the grams-not-kg slice
print(f"Quarter {q0}: {len(detail)} shipments, "
      f"{len(suspect)} with implausible (grams) carbon values")
print("Disclosed vs recomputed delta by quarter:")
rep.reconciliation[["quarter", "disclosed_carbon_kg", "recomputed_carbon_kg", "delta_kg"]]


### 8. Metric definitions & traceability assessment

**Shared metric definition — quarterly carbon:** sum of per-shipment `carbon_kg` over all
non-cancelled orders in the quarter, **normalized to kilograms**, attributed to the order's
quarter by `order_ts`. This definition must be central (MP1) so Orders and Inventory report
one number.

**Where traceability holds:** every disclosed quarterly figure decomposes to per-order, then
per-shipment rows via lineage (`quarterly_sustainability` ← `order_carbon` ← `shipment` +
`orders`). Any reported number can be drilled to source.

**Where it breaks:** the `shipment.carbon_kg` column mixes units — a ~3% slice is stored in
grams. The disclosed figure (summing the column blindly) is **inflated** vs the recomputed,
unit-normalized figure; the reconciliation table shows a non-zero `delta_kg` every quarter.
Traceability technically holds (you *can* find the rows) but **integrity** breaks at the
unit boundary: the number is auditable to a wrong source value. The fix is the Step 4
downstream normalization plus a Step 6 monitor (`carbon_kg > 100`) so the defect alarms
before it reaches a disclosure.


## Step 9 — Run governance as an operation (MP9)

Governance is ongoing, not a project. Run the provided governance audit (it rolls up the
signals from Steps 2–8: missing metadata, contract validity, quality failures, lineage
gaps, and **access-control gaps**). Investigate the findings, **remediate the fixable ones
in AWS, and re-run the audit until the platform is compliant** — then specify an operations
dashboard and a new-domain onboarding checklist for everything that remains an ongoing watch.


In [ ]:
findings = gc.governance_audit(contracts_valid=True)
print("Total findings:", len(findings))
display(findings.groupby(["signal", "severity"]).size().reset_index(name="count"))
findings.head(20)


In [ ]:
# Triage: surface the high-severity findings and propose an owner per signal.
owner_map = {
    "missing_metadata": "owning domain lead",
    "untagged_column": "Central Governance Lead",
    "quality_violation": "owning domain lead",
    "missing_lineage": "Platform Engineering",
    "consent_conflict": "Customer Identity Lead",
    "contract_invalid": "owning domain lead",
}
high = findings[findings["severity"] == "high"].copy()
high["owner"] = high["signal"].map(owner_map)
high[["signal", "object", "severity", "owner"]]


**Remediate the access-control gap in AWS, then re-run.** The audit flagged a PII table that
Step 7 left ungoverned (`campaign_membership` holds `cust_ref` = customer email). Bring it
under Lake Formation — tag it `governed=true` and deny the PII column to the analyst — then
re-run the audit and confirm the access-control finding clears. The residual findings
(quality and consent drift) are recurring SLOs you track on the dashboard, not one-time fixes.


In [ ]:
# Close the access-control gap on the other PII table, in Lake Formation.
gc.assign_tag("campaign_membership", "governed", "true")                 # tag-based coverage
gc.grant_columns("co_marketing_analyst", "campaign_membership",
                 exclude=["cust_ref"])                                    # deny the PII column
re_findings = gc.governance_audit(contracts_valid=True)
before = int((findings["signal"] == "access_control_gap").sum())
after = int((re_findings["signal"] == "access_control_gap").sum())
print(f"access_control_gap: {before} -> {after}")
assert after == 0, "access-control gap still open"
re_findings.groupby(["signal", "severity"]).size().reset_index(name="count")


### 9. Governance operations dashboard + onboarding checklist

**Dashboard specification** (each signal → source → threshold → owner):

| Panel | Signal | Source | Threshold (alert) | Owner |
|---|---|---|---|---|
| Catalog completeness | tables missing owner/classification/retention | `catalog_audit` (MP2) | any > 0 | domain leads |
| Untagged PII | columns holding PII with no classification | `catalog_audit` (MP2) | any > 0 | Central Governance Lead |
| Contract conformance | products failing the validator | validator (MP3) | any invalid | domain leads |
| Quality SLO breach | failing expectations by dimension | `quality_suite` (MP4) | > SLO threshold | domain leads |
| Consent consistency | loyalty≠marketing consent rate | semantic conflict (MP2/MP5) | > 1% | Customer Identity Lead |
| Lineage coverage | datasets with no lineage events | lineage (MP6) | any > 0 | Platform Engineering |
| Carbon integrity | out-of-range carbon values | report recon (MP8) | delta > 0 | CSRD Owner |

**New-domain onboarding checklist** (matches what the Terraform baseline enforces):
1. Register the domain's S3 location with Lake Formation (no raw S3 grants).
2. Catalog every table with `owner`, `classification`, `retention` set — gate publish on it.
3. Tag PII columns to the platform standard; apply `governed=true` LF-Tag.
4. Author a contract per data product; it must pass the validator.
5. Emit OpenLineage events from every job.
6. Read consent from the loyalty SoR — never duplicate it.
7. Subscribe the domain's signals to the dashboard above.

**Governance debt, before/after:** at intake the platform had 14 catalog gaps, 8 failing
quality expectations, 3 unresolved semantic conflicts, and 0% lineage governance gating;
after this project, PII is tagged, contracts gate publish, access is enforced by LF, and the
carbon defect is monitored — the residual debt is the structural shared-key fix (Step 5) and
the recurring consent drift, both now tracked on the dashboard.


## Step 10 — Architect's Brief (final deliverable)

Write a 1–2 page brief for Cardinal's leadership: the governance posture you established,
the major design decisions and their tradeoffs, and a next-quarter roadmap. Tie each
recommendation to specific evidence from the steps above.


### Architect's Brief — Cardinal Outfitters Federated Data Governance

**Posture.** I established **federated governance with a thin central core**: domains keep
ownership of their data products while four decisions — PII classification, the consent
system of record, the contract schema, and shared metric definitions — are centralized and
gate publication. This satisfies the GDPR and CSRD obligations that cross every domain
without seizing day-to-day control from four independently-shipping teams.

**What we found and fixed.** The catalog audit surfaced 14 governance gaps, including an
untagged PII column (`customer.phone`) and an entirely ungoverned PII table
(`campaign_membership`). The quality suite failed 8 of 9 expectations across all six
dimensions; I triaged them into quarantine (invalid contact PII), source fixes (mandatory
attributes, negative amounts, duplicated consent), and downstream fixes (carbon units,
staleness). Master-data resolution found duplicate customers created by marketing keying on
email instead of the loyalty id. Lake Formation now enforces column-, row-, tag-, and
metadata-level access across four personas, verified by querying as each. The CSRD pipeline
is traceable end-to-end, and I caught a unit defect inflating the disclosure.

**Key tradeoffs.** (1) Federated over centralized: accepts slower standard-setting in
exchange for domain buy-in and no bottleneck. (2) MDM survivorship favors *false negatives
over false positives* — I leave uncertain duplicates unmerged because exposing one person's
data to another is the worse GDPR error. (3) Access is least-privilege to a fault for the
auditor (metadata-only), trading convenience for a defensible audit posture.

**Next-quarter roadmap.** (1) Eliminate email-based customer resolution by propagating the
loyalty `customer_id` into the marketing domain — the structural fix for both duplication
and consent drift. (2) Move the manual Step 7 grants to the `governed-data-product`
Terraform module so every product is governed identically. (3) Stand up the Step 9 dashboard
and wire the recurring signals (consent rate, carbon integrity, catalog completeness) to
owners with thresholds. (4) Make contract validation a hard publish gate in CI.

**Evidence trail:** every claim above maps to a step in this notebook — the audit (2), the
contract validator (3), the quality suite (4), the resolution log (5), the lineage trace
(6), the verification table (7), the reconciliation (8), and the governance audit (9).


## Appendix — Reset
Re-baseline Lake Formation (drop all grants/tags/filters) if you want to redo Step 7 from a clean state. Re-running the whole notebook also rebuilds the platform from scratch.

In [ ]:
gc.reset()
print("Lake Formation re-baselined. Current grants:")
gc.grants_summary()
